# v7 SIMILAR languages, continual (GPU 0): Hindi → Marathi → Nepali → Hindi → ...
A new model from scratch, same recipe as the distant-language continual run, but the three languages are closely related and share one script (Devanagari). Output is restricted to Devanagari for all three.
New tokenizer `tokenizer_v7` (built the same way as tokenizer_v4, on English + Hindi + Marathi + Nepali), with tags `<2hi>`, `<2mr>`, `<2ne>`.
Question: does switching between SIMILAR languages disrupt the self as much as switching between distant ones?

**Just Run All.** The first cell prepares the data and the tokenizer (about 15-25 min the first time). Rerunning resumes from the last saved phase.

In [1]:
import os, sys, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CML_LANGSET"] = "similar"           # Hindi, Marathi, Nepali + tokenizer_v7 (must be set before importing common_v7)
os.environ["HF_HUB_DISABLE_XET"] = "1"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
# 1) Marathi + Nepali data, FLORES, and tokenizer_v7 (skips what already exists)
subprocess.run([sys.executable, "-u", "prep_similar_data.py"], check=True)

Generating train split: 100%|██████████| 3627480/3627480 [00:03<00:00, 1193054.74 examples/s]


wrote data/en-mr/val.jsonl 2000
wrote data/en-mr/train.jsonl 1000000


Generating validation split: 100%|██████████| 2000/2000 [00:00<00:00, 648821.10 examples/s]


wrote data/en-ne/train.jsonl 127074
wrote data/en-ne/val.jsonl 1413


Generating dev split: 997 examples [00:00, 142371.00 examples/s]
Generating devtest split: 1012 examples [00:00, 159776.99 examples/s]


wrote data/flores/dev.mr.jsonl 997
wrote data/flores/devtest.mr.jsonl 1012


Generating dev split: 997 examples [00:00, 131637.27 examples/s]
Generating devtest split: 1012 examples [00:00, 166338.88 examples/s]


wrote data/flores/dev.ne.jsonl 997
wrote data/flores/devtest.ne.jsonl 1012
tokenizer text en: 160,979 lines, 20,000,313 characters
tokenizer text hi: 168,705 lines, 20,000,181 characters
tokenizer text mr: 234,161 lines, 20,000,120 characters
tokenizer text ne: 182,415 lines, 13,853,411 characters


I0000 00:00:1793689185.538114    6136 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: data/tok_train_v7.txt
  input_format: 
  model_prefix: tokenizer_v7/spm
  model_type: UNIGRAM
  vocab_size: 32000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 4000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4096
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <2hi>
  user_defined_symbols: <2mr>
  user_defined_symbols: <2ne>
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  

  hi: 1.10 pieces per word | ▁सरकार ▁ने ▁किसानों ▁के ▁लिए ▁नई ▁योजना ▁की ▁घोषणा ▁की ।
  mr: 1.30 pieces per word | ▁त्याच्या ▁घर ाजवळ ▁एक ▁मोठे ▁तळ े ▁आहे ▁आणि ▁तिथे ▁पक्षी ▁येतात .
  ne: 1.43 pieces per word | ▁सरकारले ▁किसान हरूका ▁लागि ▁नयाँ ▁योजना को ▁घोषणा ▁गर्‍यो ।
data ready: {'hi': 993016, 'mr': 1000000, 'ne': 127074}


CompletedProcess(args=['/venv/main/bin/python', '-u', 'prep_similar_data.py'], returncode=0)

In [2]:
# 2) load the training code (uses tokenizer_v7 because of CML_LANGSET)
import glob, json, time
from common_v7 import *
print("languages:", LANGS, "| tags:", LANG_TAG, "| vocab:", V, "| device:", DEVICE)
for l, s_ in [("hi", "सरकार ने किसानों के लिए नई योजना की घोषणा की।"), ("mr", "त्याच्या घराजवळ एक मोठे तळे आहे."), ("ne", "सरकारले नयाँ योजनाको घोषणा गर्‍यो।")]:
    print(f"  {l}: {len(sp.encode(s_)) / len(s_.split()):.2f} pieces per word")
RUN = "v7_sim_continual_s0"
from huggingface_hub import HfApi
HF_REPO = HfApi().whoami()["name"] + "/cml-lang-self-checkpoints"

languages: ['hi', 'mr', 'ne'] | tags: {'hi': 4, 'mr': 5, 'ne': 6} | vocab: 32000 | device: cuda
  hi: 1.10 pieces per word
  mr: 1.50 pieces per word
  ne: 1.40 pieces per word


In [3]:
C = RunConfig(run_name=RUN, mode="continual", order=("hi", "mr", "ne"), n_phases=60, seed=0, time_cap_h=120.0, hf_repo=HF_REPO)
print(C)

RunConfig(run_name='v7_sim_continual_s0', mode='continual', order=('hi', 'mr', 'ne'), init_ckpt=None, n_phases=60, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints', start_phase=None, noise_per_step=0.0, boundary_push=0.0, health_every=1, redo_every=0, redo_thresh=0.01)


In [4]:
D = load_data(train_langs=["hi", "mr", "ne"])

  hi: 993,016 train / 2,000 val
  mr: 1,000,000 train / 2,000 val
  ne: 127,074 train / 1,413 val
data loaded in 25s


In [5]:
model = train_run(C, D)

 RUN v7_sim_continual_s0 | mode continual | 60 phases after phase 0 | device cuda
 phase order: init (random) → hi → mr → ne → hi → mr → ne → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
phase 0: fresh random model (66.8M params)

 PHASE 0 DONE | trained: init | starting model, no training
--------------------------------------------------------------------------------------
   lang      val       Δ |   chrF      Δ |  chrF (unrestricted)  val (unrestr.)
   hi     10.149         |    2.0        |                  0.5          10.529
   mr     10.141         |    2.8        |                  1.4          10.522
   ne     10.155         |    2.3        |                  2.0          10.546

▶ PHASE 1/60 | training hi (993,016 pairs)


p000_init.pt:   0%|          | 0.00/267M [00:00<?, ?B/s]

  [p01 hi] step    500 | lr 2.0e-04 | train 7.218 | val 5.404 (best 5.404) | window Δ n/a | stable | 77k tok/s | phase 1m37s | run ETA: after first phase | health {'wnorm': 144.378, 'rank_enc1': 14.2, 'rank_dec1': 25.6, 'rarely_active': 0.0064}
  [p01 hi] step  1,000 | lr 2.0e-04 | train 5.607 | val 4.408 (best 4.408) | window Δ n/a | stable | 79k tok/s | phase 3m09s | run ETA: after first phase | health {'wnorm': 147.04, 'rank_enc1': 27.0, 'rank_dec1': 75.1, 'rarely_active': 0.0004}


KeyboardInterrupt: 

In [ ]:
print_run_summary(RUN)

In [1]:
!nvidia-smi

Tue Nov  3 07:23:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:2D:00.0 Off |                  Off |
|  0%   58C    P8             38W /  350W |      47MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----